### Building an end-to-end data engineering pipeline that ingests, cleans, standardizes, and transforms flight data into a reliable analytical dataset for reporting and business intelligence purposes - Neostats: ASG Airlines


### Importing required libraries for the solution


In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
import logging

logging.basicConfig(
    filename="airline_pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

### Loading the provided Excel(.xlsx) file - "UseCase_Airline.xlsx"

In [3]:
file_path = os.getenv(
    "FLIGHT_DATA_FILE",
    "UseCase_Airlines.xlsx"
)

flights = pd.read_excel(file_path,sheet_name="flights")

bookings = pd.read_excel(file_path,sheet_name="bookings")

passengers = pd.read_excel(file_path,sheet_name="passengers")

payments = pd.read_excel(file_path,sheet_name="payments")

source_flight_count = len(flights)
print("Loaded the Excel file successfully.")
print("Rows and columns in each sheet:")
print("Flights:", flights.shape)
print("Bookings:", bookings.shape)
print("Passengers:", passengers.shape)
print("Payments:", payments.shape)
logger.info("Excel file loaded successfully.")

Loaded the Excel file successfully.
Rows and columns in each sheet:
Flights: (1020, 7)
Bookings: (1000, 9)
Passengers: (1039, 9)
Payments: (1000, 4)


### Function to standardize columns

In [4]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

### Standardizing the column names

In [5]:
flights = standardize_columns(flights)
bookings = standardize_columns(bookings)
passengers = standardize_columns(passengers)
payments = standardize_columns(payments)

print("Column names standardized for whole excel file.")

Column names standardized for whole excel file.


### Validating the "Flights" Schema (Quality check during ingestion)

If the source file changes and one of the expected fields disappears, the pipeline stops immediately instead of producing incorrect results.

In [6]:
required_columns = {
    "flight_id",
    "airline",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "duration"
}

missing_columns = required_columns - set(flights.columns)

if missing_columns:
    logger.error("Missing required flight columns: %s", missing_columns)
    raise ValueError(f"There are missing required columns from the Flight Schema: {missing_columns}")

print("Flight schema validation passed. All required flight columns are present.")
logger.info("Flight schema validation passed.")

Flight schema validation passed. All required flight columns are present.


### Data Profiling before Cleaning

In [7]:
missing_values = flights.isnull().sum()

print("Missing values in flight dataset for each column:")
print(missing_values)

Missing values in flight dataset for each column:
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64


In [8]:
flights["flight_id"] = (
    flights["flight_id"].astype("string").str.strip().str.upper()
)

flights["airline"] = (
    flights["airline"].astype("string").str.strip().str.upper()
)

flights["source"] = (
    flights["source"].astype("string").str.strip().str.upper()
)

flights["destination"] = (
    flights["destination"].astype("string").str.strip().str.upper()
)
print ("Standardized flight values from the flights dataset.")

Standardized flight values from the flights dataset.


### Derving missing/unknown "airline" using "flight_id" by using the following pattern identified from the dataset

In [9]:
airline_from_flight_id = {
    "6F": "INDIGO",
    "AI": "AIR INDIA",
    "SJ": "SPICEJET",
    "UK": "VISTARA"
}

unknown_airline = (
    flights["airline"].isna() | flights["airline"].isin(["UNKNOWN"]) #Storing all unknown/missing airlines in a variable
) 

print("Missing/unknown airline records before correction:",
      unknown_airline.sum())


def derive_airline(flight_id):
    if pd.isna(flight_id):
        return "UNKNOWN"

    prefix = str(flight_id)[:2].upper()
    return airline_from_flight_id.get(prefix, "UNKNOWN")


flights.loc[unknown_airline, "airline"] = (
    flights.loc[unknown_airline, "flight_id"]
    .apply(derive_airline)
)

print("Airline values derived from flight ID prefixes.")

print("Airline distribution after correction:")
print(flights["airline"].value_counts().to_string())

print("Remaining UNKNOWN airlines:",
      (flights["airline"] == "UNKNOWN").sum())

Missing/unknown airline records before correction: 72
Airline values derived from flight ID prefixes.
Airline distribution after correction:
airline
INDIGO       273
AIR INDIA    260
SPICEJET     251
VISTARA      236
Remaining UNKNOWN airlines: 0


### Converting the values(arrival_time & departure_time) into proper datetime data type

In [10]:
flights["departure_time"] = pd.to_datetime(
    flights["departure_time"],
    errors="coerce"
)

flights["arrival_time"] = pd.to_datetime(
    flights["arrival_time"],
    errors="coerce"
)

print("Departure and arrival times converted to datetime format.")
# Counting and printing the invalid timestamps
print("Invalid departure timestamps:", flights["departure_time"].isna().sum())
print("Invalid arrival timestamps:", flights["arrival_time"].isna().sum())

Departure and arrival times converted to datetime format.
Invalid departure timestamps: 0
Invalid arrival timestamps: 0


### Calculating number of overnight flights

In [11]:
flights["is_overnight"] = (
    flights["arrival_time"].notna() & flights["departure_time"].notna()
    & 
    (flights["arrival_time"].dt.date > flights["departure_time"].dt.date)
)

print("Overnight flights:", flights["is_overnight"].sum())

Overnight flights: 124


#### Calculating flight duration in minutes

In [12]:
flights["calculated_duration_minutes"] = (
    flights["arrival_time"] - flights["departure_time"]
).dt.total_seconds() / 60

# Keeping the calculated duration in minutes rounded to only 2 decimal places
flights["calculated_duration_minutes"] = (
    flights["calculated_duration_minutes"].round(2)
)
print("Flight duration calculated from departure and arrival timestamps.")
# Prints the minimum and maximum duration in the whole flight dataset 
print(
    "Calculated duration range:",
    flights["calculated_duration_minutes"].min(),
    "to",
    flights["calculated_duration_minutes"].max(),
    "minutes"
)

Flight duration calculated from departure and arrival timestamps.
Calculated duration range: -1140.0 to 300.0 minutes


### Filtering all flights with a negative calculated_duration_minutes

In [13]:
negative_duration = flights[
    flights["calculated_duration_minutes"] < 0
][[
    "flight_id",
    "airline",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "duration",
    "calculated_duration_minutes"
]]
# Printing all flights detail with a negative calculated_duration_minutes indicating invalid timestamps
print("Negative duration records:", len(negative_duration))
negative_duration

Negative duration records: 1


,flight_id,airline,source,destination,departure_time,arrival_time,duration,calculated_duration_minutes
355,SJ192,SPICEJET,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,1899-12-29 05:00:00,-1140.0


### Checking for time anomaly in the dateset

In [14]:
# Created a new column: if departure_time or arrival_time is missing, or if arrival_time is earlier than departure_time then it's a time anomaly
flights["time_anomaly"] = (
    flights["departure_time"].isna()
    | flights["arrival_time"].isna()
    | (
        flights["arrival_time"].notna()
        & flights["departure_time"].notna()
        & (flights["arrival_time"] < flights["departure_time"])
    )
)
# Printing number of flights with time anomalies in the dataset
print("Time anomalies:", flights["time_anomaly"].sum())

Time anomalies: 1


### Converting the original flights dateset columnn:"duration" into minutes.

In [15]:
# Function to convert duration to minutes
def duration_to_minutes(value):
    if pd.isna(value):
        return np.nan

    if hasattr(value, "hour"):
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
        )

    return np.nan

# Applying the function to the entire column:"duration"
flights["duration"] = flights["duration"].apply(
    duration_to_minutes
).round(2)

print("Source duration converted to minutes.")
print(
    "Missing/invalid durations:",
    flights["duration"].isna().sum()
)
#Checks for duration mismatches i.e. if source duration is different from the duration calculated from the timestamps
flights["duration_mismatch"] = (
    flights["duration"].notna()
    & flights["calculated_duration_minutes"].notna()
    & (
        flights["duration"].round(2)
        != flights["calculated_duration_minutes"].round(2)
    )
)
#printing the number of duration mismatches in the dataset
print(
    "Duration mismatches:",
    flights["duration_mismatch"].sum()
)

Source duration converted to minutes.
Missing/invalid durations: 0
Duration mismatches: 1


### Creating a final-standardized duration column in the dataste


In [16]:
flights["duration_minutes"] = (flights["calculated_duration_minutes"])

### Checking for route anomaly in the dateset

In [17]:
flights["route_anomaly"] = (
    flights["source"].isna()
    | flights["destination"].isna()
    | (
        flights["source"].notna()
        & flights["destination"].notna()
        & (flights["source"] == flights["destination"])
    )
)
#printing route anomalies count in the dataset
print("Route anomalies:", flights["route_anomaly"].sum())

Route anomalies: 0


### Checking for duration anomaly in the dateset

In [18]:
flights["duration_anomaly"] = (
    flights["duration_minutes"].isna()
    | (flights["duration_minutes"] <= 0)
)
#printing duration anomalies count in the dataset
print("Duration anomalies:", flights["duration_anomaly"].sum())

Duration anomalies: 1


### final validation step to check if flight is valid or not

In [19]:
flights["is_valid"] = ~(
    flights["flight_id"].isna() | flights["time_anomaly"]| flights["route_anomaly"]| flights["duration_anomaly"])
# Print number of valid & invalid records
print("Valid records:", flights["is_valid"].sum())
print("Invalid records:", (~flights["is_valid"]).sum())

Valid records: 1019
Invalid records: 1


### finding duplicate flight records based on the combination of "flight_id" and "departure_time".

In [20]:
duplicate_mask = flights.duplicated(
    subset=["flight_id", "departure_time"],
    keep=False
)
duplicate_count = duplicate_mask.sum()
print("Duplicate flight records:", duplicate_count)
duplicates = flights[duplicate_mask].sort_values(
    ["flight_id", "departure_time"]
)
#printing duplicate flight details in the dataset
print("Duplicate flight details")
display(duplicates)

# Removing the duplicate entries and keeping the first occurrence only
duplicate_count = flights.duplicated(
    subset=["flight_id", "departure_time"],
    keep="first"
).sum()

print("Duplicate records to be removed:", duplicate_count)

flights = flights.drop_duplicates(
    subset=["flight_id", "departure_time"],
    keep="first"
).copy()

print("Duplicate records removed successfully.")
logger.info("Flight duplicate removal completed. Records removed: %d", duplicate_count)
print("Remaining flight records:", len(flights))

Duplicate flight records: 30
Duplicate flight details


,flight_id,airline,source,destination,departure_time,arrival_time,duration,is_overnight,calculated_duration_minutes,time_anomaly,duration_mismatch,duration_minutes,route_anomaly,duration_anomaly,is_valid
549,AI020,AIR INDIA,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,148.0,False,148.0,False,False,148.0,False,False,True
550,AI020,AIR INDIA,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,148.0,False,148.0,False,False,148.0,False,False,True
141,AI031,AIR INDIA,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,186.0,False,186.0,False,False,186.0,False,False,True
142,AI031,AIR INDIA,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,186.0,False,186.0,False,False,186.0,False,False,True
598,AI043,AIR INDIA,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,249.0,False,249.0,False,False,249.0,False,False,True
599,AI043,AIR INDIA,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,249.0,False,249.0,False,False,249.0,False,False,True
602,AI070,AIR INDIA,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,99.0,False,99.0,False,False,99.0,False,False,True
603,AI070,AIR INDIA,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,99.0,False,99.0,False,False,99.0,False,False,True
83,AI242,AIR INDIA,BLR,CCU,2026-04-20 16:41:41.704,2026-04-20 17:43:41.704,62.0,False,62.0,False,False,62.0,False,False,True
84,AI242,AIR INDIA,BLR,CCU,2026-04-20 16:41:41.704,2026-04-20 17:43:41.704,62.0,False,62.0,False,False,62.0,False,False,True


Duplicate records to be removed: 15
Duplicate records removed successfully.
Remaining flight records: 1005


### Splitting the dataset into valid and invalid flight records

In [21]:
clean_flights = flights[flights["is_valid"]].copy()

rejected_flights = flights[~flights["is_valid"]].copy()

clean_count = len(clean_flights)
rejected_count = len(rejected_flights)
total_count = len(flights)

print("Clean flight records:", clean_count)
print("Rejected/anomalous flight records:", rejected_count)
print("Total records:", total_count)
logger.warning("Rejected flight records: %d", rejected_count)
if clean_count + rejected_count == total_count:
    print("No records were lost.")
else:
    raise ValueError("record counts do not match & validation failed.")

Clean flight records: 1004
Rejected/anomalous flight records: 1
Total records: 1005
No records were lost.


### assigning a clear reason to every rejected flight entries

In [22]:
rejected_flights["rejection_reason"] = np.select(
    [
        rejected_flights["time_anomaly"],
        rejected_flights["route_anomaly"],
        rejected_flights["duration_anomaly"]
    ],
    [
        "Invalid departure/arrival timestamps",
        "Invalid source/destination",
        "Invalid or missing duration"
    ],
    default="Other data quality issue"
)
#printing flight_id with its rejection detail
print("Rejection reasons assigned successfully.")
print(rejected_flights[["flight_id", "rejection_reason"]].to_string(index=False))

Rejection reasons assigned successfully.
flight_id                     rejection_reason
    SJ192 Invalid departure/arrival timestamps


### Final validation of clean flight dataset

In [23]:
print("Total clean records:", len(clean_flights))
print("Invalid records remaining:", (~clean_flights["is_valid"]).sum())

if (~clean_flights["is_valid"]).sum() == 0:
    print("Validation passed: clean_flights contains only valid records.")
else:
    raise ValueError("Validation failed: invalid records remain in clean_flights.")

Total clean records: 1004
Invalid records remaining: 0
Validation passed: clean_flights contains only valid records.


In [24]:
missing_values = clean_flights.isnull().sum()

print("Missing values in clean flight dataset:")
print(missing_values)

if missing_values.sum() == 0:
    print(" No missing values remain.")
else:
    print("Missing values are present and require review- validation not passed.")

Missing values in clean flight dataset:
flight_id                      0
airline                        0
source                         0
destination                    0
departure_time                 0
arrival_time                   0
duration                       0
is_overnight                   0
calculated_duration_minutes    0
time_anomaly                   0
duration_mismatch              0
duration_minutes               0
route_anomaly                  0
duration_anomaly               0
is_valid                       0
dtype: int64
 No missing values remain.


### PASSENGERS

### Passengers data quality check

In [25]:
passenger_missing = passengers.isnull().sum()

print("Missing values in passenger dataset:")
print(passenger_missing)

print("\nTotal missing values:",
      passenger_missing.sum())

Missing values in passenger dataset:
passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64

Total missing values: 10


In [26]:
### Checking duplicate passenger records

passenger_duplicate_mask = passengers.duplicated(
    subset=["passenger_id"],
    keep=False
)

print("Duplicate passenger records:",
      passenger_duplicate_mask.sum())

passengers = passengers.drop_duplicates(
    subset=["passenger_id"],
    keep="first"
).copy()

print("Duplicate passenger records removed.")
print("Remaining passenger records:", len(passengers))

Duplicate passenger records: 75
Duplicate passenger records removed.
Remaining passenger records: 1000


In [27]:
### Validating passenger records

invalid_passenger_mask = (
    passengers["passenger_id"].isna()
)

clean_passengers = passengers[
    ~invalid_passenger_mask
].copy()

rejected_passengers = passengers[
    invalid_passenger_mask
].copy()

print("Clean passenger records:", len(clean_passengers))
print("Rejected passenger records:", len(rejected_passengers))

Clean passenger records: 1000
Rejected passenger records: 0


### PII Masking Technique: Data masking 

In [28]:
print("Passenger data privacy check")

print("Clean passenger dataset:")
print("Columns:", clean_passengers.columns.tolist())
print("Shape:", clean_passengers.shape)

Passenger data privacy check
Clean passenger dataset:
Columns: ['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']
Shape: (1000, 9)


### Masking passenger PII

In [29]:
masked_passengers = clean_passengers.copy()

masked_passengers["first_name"] = "MASKED"
masked_passengers["last_name"] = "MASKED"
masked_passengers["email"] = "MASKED"
masked_passengers["phone"] = "MASKED"
masked_passengers["aadhaar_id"] = "MASKED"

# Masking the DOB while retaining the year only
masked_passengers["date_of_birth"] = pd.to_datetime(
    masked_passengers["date_of_birth"],
    errors="coerce"
).dt.year

print("PII masking completed successfully.")

print("Masked passenger columns:")
print(masked_passengers.columns.tolist())

print("Sample masked records:")
print(masked_passengers.head().to_string(index=False))

PII masking completed successfully.
Masked passenger columns:
['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']
Sample masked records:
passenger_id first_name last_name  age gender  email  phone aadhaar_id  date_of_birth
       P1000     MASKED    MASKED   52      F MASKED MASKED     MASKED           1974
       P1001     MASKED    MASKED   15      M MASKED MASKED     MASKED           2011
       P1002     MASKED    MASKED   72      M MASKED MASKED     MASKED           1954
       P1003     MASKED    MASKED   61      F MASKED MASKED     MASKED           1965
       P1004     MASKED    MASKED   21      M MASKED MASKED     MASKED           2005


### PII masking validation

In [30]:
pii_columns = [
    "first_name",
    "last_name",
    "email",
    "phone",
    "aadhaar_id"
]

for column in pii_columns:
    masked_count = (masked_passengers[column] == "MASKED").sum()
    print(f"{column}: {masked_count} records masked")

if all((masked_passengers[column] == "MASKED").all() for column in pii_columns):
    print("\nPII validation passed: all direct PII fields are masked.")
else:
    raise ValueError("PII validation failed: unmasked PII remains.")

first_name: 1000 records masked
last_name: 1000 records masked
email: 1000 records masked
phone: 1000 records masked
aadhaar_id: 1000 records masked

PII validation passed: all direct PII fields are masked.


In [31]:
masked_passengers.to_csv(
    "masked_passengers.csv",
    index=False
)

print("Masked passenger dataset saved successfully.")
print("File: masked_passengers.csv")
print("Records:", len(masked_passengers))
print("Purpose: de-identified passenger data for safe analytical use.")

Masked passenger dataset saved successfully.
File: masked_passengers.csv
Records: 1000
Purpose: de-identified passenger data for safe analytical use.


### Booking


### Booking data quality check

In [32]:
booking_missing = bookings.isnull().sum()

print("Missing values in booking dataset:")
print(booking_missing)

print("Total missing values:",booking_missing.sum())

Missing values in booking dataset:
booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64
Total missing values: 45


### Standardizing booking values


In [33]:

bookings["booking_id"] = (
    bookings["booking_id"].astype("string").str.strip().str.upper()
)

bookings["passenger_id"] = (
    bookings["passenger_id"].astype("string").str.strip().str.upper()
)

bookings["flight_id"] = (
    bookings["flight_id"].astype("string").str.strip().str.upper()
)

bookings["status"] = (
    bookings["status"].astype("string").str.strip().str.upper()
)

bookings["seat_number"] = (
    bookings["seat_number"].astype("string").str.strip().str.upper()
)

print("Booking values standardized successfully.")

Booking values standardized successfully.


### Converting Booking data

In [34]:
bookings["booking_date"] = pd.to_datetime(
    bookings["booking_date"],
    errors="coerce"
)

invalid_booking_dates = bookings["booking_date"].isna().sum()

print("Invalid booking dates:", invalid_booking_dates)

Invalid booking dates: 0


In [35]:
print("Booking status values:")
print(bookings["status"].value_counts(dropna=False).to_string())

Booking status values:
status
CONFIRMED    320
CANCELLED    314
PENDING      291
<NA>          45
INVALID       30


### Checking duplicate entries in booking

In [36]:
booking_duplicate_mask = bookings.duplicated(
    subset=["booking_id"],
    keep=False
)

print(
    "Duplicate booking records:",
    booking_duplicate_mask.sum()
)

bookings = bookings.drop_duplicates(
    subset=["booking_id"],
    keep="first"
).copy()

print("Duplicate booking records removed.")
print("Remaining booking records:", len(bookings))

Duplicate booking records: 0
Duplicate booking records removed.
Remaining booking records: 1000


In [37]:
### Checking booking-to-flight referential integrity

bookings["fk_flight_valid"] = (
    bookings["flight_id"].isin(clean_flights["flight_id"])
)

print("Bookings with valid flight references:",
      bookings["fk_flight_valid"].sum())

print("Bookings with invalid flight references:",
      (~bookings["fk_flight_valid"]).sum())


### Validating booking records

invalid_booking_mask = (
    bookings["booking_id"].isna()
    | bookings["flight_id"].isna()
    | bookings["status"].isna()
    | bookings["booking_date"].isna()
    | (bookings["status"] == "INVALID")
    | (~bookings["fk_flight_valid"])
)

clean_bookings = bookings[
    ~invalid_booking_mask
].copy()

rejected_bookings = bookings[
    invalid_booking_mask
].copy()

print("Clean booking records:", len(clean_bookings))
print("Rejected booking records:", len(rejected_bookings))
logger.warning("Rejected booking records: %d", len(rejected_bookings))

if len(clean_bookings) + len(rejected_bookings) == len(bookings):
    print("Booking validation completed successfully.")
else:
    raise ValueError("Booking records were lost during validation.")

Bookings with valid flight references: 999
Bookings with invalid flight references: 1
Clean booking records: 924
Rejected booking records: 76
Booking validation completed successfully.


### Masking booking sensitive data

In [38]:
masked_bookings = clean_bookings.copy()
# Mask sensitive booking fields
masked_bookings["passenger_id"] = "MASKED"
masked_bookings["passport_number"] = "MASKED"
masked_bookings["emergency_contact_name"] = "MASKED"
masked_bookings["emergency_contact_phone"] = "MASKED"

print("Booking sensitive fields masked successfully.")

print("Masked booking columns:")
print(masked_bookings.columns.tolist())

print("Sample masked booking records:")
print(masked_bookings.head().to_string(index=False))


Booking sensitive fields masked successfully.
Masked booking columns:
['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone', 'fk_flight_valid']
Sample masked booking records:
booking_id passenger_id flight_id            booking_date    status passport_number seat_number emergency_contact_name emergency_contact_phone  fk_flight_valid
     B1000       MASKED     AI192 2025-06-14 11:37:36.951 CANCELLED          MASKED          3D                 MASKED                  MASKED             True
     B1001       MASKED     6F026 2025-11-02 11:37:36.951 CANCELLED          MASKED         18A                 MASKED                  MASKED             True
     B1002       MASKED     SJ010 2025-08-25 11:37:36.951 CANCELLED          MASKED         30C                 MASKED                  MASKED             True
     B1003       MASKED     AI069 2025-12-30 11:37:36.951 CONFIRMED          MASKED

### Validating booking data masking

In [39]:
booking_pii_columns = [
    "passenger_id",
    "passport_number",
    "emergency_contact_name",
    "emergency_contact_phone"
]

for column in booking_pii_columns:
    masked_count = (masked_bookings[column] == "MASKED").sum()
    print(f"{column}: {masked_count} records masked")

if all(
    (masked_bookings[column] == "MASKED").all()
    for column in booking_pii_columns
):
    print("Booking PII validation passed: all sensitive fields are masked.")
else:
    raise ValueError("Booking PII masking validation failed.")

passenger_id: 924 records masked
passport_number: 924 records masked
emergency_contact_name: 924 records masked
emergency_contact_phone: 924 records masked
Booking PII validation passed: all sensitive fields are masked.


### Saving masked booking dataset in .csv

In [40]:
masked_bookings.to_csv(
    "masked_bookings.csv",
    index=False
)
print("Saved: masked_bookings.csv")
print("Clean bookings:", len(clean_bookings))
print("Rejected bookings:", len(rejected_bookings))

Saved: masked_bookings.csv
Clean bookings: 924
Rejected bookings: 76


### Payments


In [41]:
payment_missing = payments.isnull().sum()

print("Missing values in payment dataset:")
print(payment_missing)

print("Total missing values:",payment_missing.sum())

Missing values in payment dataset:
payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64
Total missing values: 48


### Standardizing payment data

In [42]:
payments["payment_id"] = payments["payment_id"].astype("string").str.strip().str.upper()
payments["booking_id"] = payments["booking_id"].astype("string").str.strip().str.upper()
payments["payment_method"] = payments["payment_method"].astype("string").str.strip().str.upper()

print("Payment values standardized.")

Payment values standardized.


### Converting payment data

In [43]:
payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

print("Payment amount converted to numeric.")

Payment amount converted to numeric.


### Remove duplicates

In [44]:
payments = payments.drop_duplicates(
    subset=["payment_id"],
    keep="first"
).copy()

print("Duplicate payment records removed.")

Duplicate payment records removed.


### Checking payment to booking referential integrity

In [45]:
payments["fk_booking_valid"] = (
    payments["booking_id"].isin(clean_bookings["booking_id"])
)

print("Payments with valid booking references:",
      payments["fk_booking_valid"].sum())

print("Payments with invalid booking references:",
      (~payments["fk_booking_valid"]).sum())


### Validating payment records

invalid_payment_mask = (
    payments["payment_id"].isna()
    | payments["booking_id"].isna()
    | payments["amount"].isna()
    | (payments["amount"] < 0)
    | payments["payment_method"].isna()
    | (~payments["fk_booking_valid"])
)

clean_payments = payments[
    ~invalid_payment_mask
].copy()

rejected_payments = payments[
    invalid_payment_mask
].copy()

print("Clean payment records:", len(clean_payments))
print("Rejected payment records:", len(rejected_payments))
logger.warning("Rejected payment records: %d", len(rejected_payments))

Payments with valid booking references: 936
Payments with invalid booking references: 64
Clean payment records: 863
Rejected payment records: 137


### Saving clean and rejected payment entries in respective .csv file


In [46]:
clean_payments.to_csv("clean_payments.csv", index=False)
rejected_payments.to_csv("rejected_payments.csv", index=False)

print("Saved: clean_payments.csv")
print("Saved: rejected_payments.csv")

Saved: clean_payments.csv
Saved: rejected_payments.csv


### Printing all clean dataset details that will be used for the analysis

In [47]:
print("Flights:")
print("Shape:", clean_flights.shape)
print("Columns:", clean_flights.columns.tolist())

print("Passengers:")
print("Shape:", clean_passengers.shape)
print("Columns:", clean_passengers.columns.tolist())

print("Bookings:")
print("Shape:", clean_bookings.shape)
print("Columns:", clean_bookings.columns.tolist())

print("Payments:")
print("Shape:", clean_payments.shape)
print("Columns:", clean_payments.columns.tolist())

Flights:
Shape: (1004, 15)
Columns: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration', 'is_overnight', 'calculated_duration_minutes', 'time_anomaly', 'duration_mismatch', 'duration_minutes', 'route_anomaly', 'duration_anomaly', 'is_valid']
Passengers:
Shape: (1000, 9)
Columns: ['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']
Bookings:
Shape: (924, 10)
Columns: ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone', 'fk_flight_valid']
Payments:
Shape: (863, 5)
Columns: ['payment_id', 'booking_id', 'amount', 'payment_method', 'fk_booking_valid']


### Flights KPI 

In [48]:
analysis_flights = clean_flights[
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "departure_time",
        "arrival_time",
        "is_overnight",
        "duration_minutes"
    ]
].copy()

print("Final analysis dataset created successfully.")
print("Records:", len(analysis_flights))
print("Columns:", analysis_flights.columns.tolist())

Final analysis dataset created successfully.
Records: 1004
Columns: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'is_overnight', 'duration_minutes']


### Calculating average flight duration

In [49]:
average_duration = (
    analysis_flights["duration_minutes"].mean()
    if len(analysis_flights) > 0 else 0
)

print("Average Flight Duration:", round(average_duration, 2), "minutes")

Average Flight Duration: 164.49 minutes


### Calculating route wise traffic i.e. most busy route(high number of flight count)

In [50]:
route_traffic = (
    analysis_flights
    .groupby(["source", "destination"]).size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

print("Route-wise traffic calculated successfully.")
print(route_traffic.head(10).to_string(index=False))

Route-wise traffic calculated successfully.
source destination  flight_count
   BOM         CCU            90
   CCU         DEL            72
   MAA         BLR            65
   BLR         BOM            60
   HYD         MAA            57
   DEL         HYD            54
   HYD         DEL            42
   BOM         DEL            39
   CCU         BOM            33
   DEL         BLR            29


### Airline flight count distribution

In [51]:
airline_distribution = (
    analysis_flights
    .groupby("airline").size()
    .reset_index(name="flight_count")
    .sort_values("flight_count", ascending=False)
)

print("Flight distribution by airline calculated successfully.")
print(airline_distribution.to_string(index=False))

Flight distribution by airline calculated successfully.
  airline  flight_count
   INDIGO           273
AIR INDIA           255
 SPICEJET           246
  VISTARA           230


### Booking KPIs


### Total Bookings

In [52]:
total_bookings = len(clean_bookings)
print("Total bookings:", total_bookings)

Total bookings: 924


### Booking Status Distribution

In [53]:
booking_status_distribution = (
    clean_bookings["status"]
    .value_counts()
    .reset_index()
)

booking_status_distribution.columns = ["status", "booking_count"]
print(booking_status_distribution.to_string(index=False))

   status  booking_count
CONFIRMED            319
CANCELLED            314
  PENDING            291


### Booking cancellation rate

In [54]:
cancelled_bookings = (
    clean_bookings["status"] == "CANCELLED").sum()

cancellation_rate = (
    (cancelled_bookings / total_bookings) * 100
    if total_bookings > 0 else 0
)
print("Cancelled bookings:", cancelled_bookings)
print("Cancellation rate:", round(cancellation_rate, 2), "%")

Cancelled bookings: 314
Cancellation rate: 33.98 %


### Payment KPIs


### Total Revenue

In [55]:
total_payment_revenue = clean_payments["amount"].sum()
print("Total payment revenue:", round(total_payment_revenue, 2))

Total payment revenue: 6959344.65


### Average Payment Amount

In [56]:
average_payment_amount = (
    clean_payments["amount"].mean()
    if len(clean_payments) > 0
    else 0
)
print("Average payment amount:", round(average_payment_amount, 2))

Average payment amount: 8064.13


### Paymen method distribution

In [57]:
payment_method_distribution = (
    clean_payments["payment_method"]
    .value_counts()
    .reset_index()
)
payment_method_distribution.columns = ["payment_method", "payment_count"]
print(payment_method_distribution.to_string(index=False))

payment_method  payment_count
           UPI            314
    NETBANKING            275
          CARD            274


### Passengers KPI

In [58]:
total_passengers = len(masked_passengers)
print("Total passengers:", total_passengers)

Total passengers: 1000


### All flights anomaly summary

In [59]:
anomaly_summary = {
    "Time Anomalies": flights["time_anomaly"].sum(),
    "Route Anomalies": flights["route_anomaly"].sum(),
    "Duration Anomalies": flights["duration_anomaly"].sum(),
    "Duration Mismatches": flights["duration_mismatch"].sum()
}

print("Delay/Anomaly analysis:")
for anomaly, count in anomaly_summary.items():
    print(f"{anomaly}: {count}")

Delay/Anomaly analysis:
Time Anomalies: 1
Route Anomalies: 0
Duration Anomalies: 1
Duration Mismatches: 1


### Calculating flights anomaly rate (%)


In [60]:
total_flights = len(flights)
anomalous_flights = (~flights["is_valid"]).sum()

anomaly_rate = (anomalous_flights / total_flights) * 100

print("Total flight records:", total_flights)
print("Anomalous flight records:", anomalous_flights)
print("Anomaly rate:", round(anomaly_rate, 2), "%")

Total flight records: 1005
Anomalous flight records: 1
Anomaly rate: 0.1 %


### Saving the clean and rejected flights entries in respective .csv files

In [61]:
clean_flights.to_csv("clean_flights.csv", index=False)
rejected_flights.to_csv("rejected_flights.csv", index=False)

print("Clean dataset saved as: clean_flights.csv")
print("Rejected dataset saved as: rejected_flights.csv")
print("Clean records saved:", len(clean_flights))
print("Rejected records saved:", len(rejected_flights))

Clean dataset saved as: clean_flights.csv
Rejected dataset saved as: rejected_flights.csv
Clean records saved: 1004
Rejected records saved: 1


### All KPIs summary

In [62]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Average Flight Duration",
        "Total Clean Flights",
        "Total Rejected Flights",
        "Anomaly Rate",
        "Total Bookings",
        "Booking Cancellation Rate",
        "Total Payment Revenue",
        "Average Payment Amount",
        "Total Passengers"
    ],
    "Value": [
        round(average_duration, 2),
        len(clean_flights),
        len(rejected_flights),
        round(anomaly_rate, 2),
        total_bookings,
        round(cancellation_rate, 2),
        round(total_payment_revenue, 2),
        round(average_payment_amount, 2),
        total_passengers
    ]
})

print("KPI summary created successfully.")
print(kpi_summary.to_string(index=False))

KPI summary created successfully.
                      KPI      Value
  Average Flight Duration     164.49
      Total Clean Flights    1004.00
   Total Rejected Flights       1.00
             Anomaly Rate       0.10
           Total Bookings     924.00
Booking Cancellation Rate      33.98
    Total Payment Revenue 6959344.65
   Average Payment Amount    8064.13
         Total Passengers    1000.00


### Saving all KPI and its summary in its respective .csv file

In [63]:
kpi_summary.to_csv("kpi_summary.csv", index=False)
route_traffic.to_csv("route_traffic.csv", index=False)
airline_distribution.to_csv("airline_distribution.csv", index=False)
booking_status_distribution.to_csv("booking_status_distribution.csv", index=False)
payment_method_distribution.to_csv("payment_method_distribution.csv", index=False)

print("KPI files saved successfully.")


KPI files saved successfully.


### Creating the dataset to be used in PowerBI

In [64]:
powerbi_flights = analysis_flights.copy()

print("Power BI dataset created successfully.")
print("Records:", len(powerbi_flights))
print("Columns:", powerbi_flights.columns.tolist())

Power BI dataset created successfully.
Records: 1004
Columns: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'is_overnight', 'duration_minutes']


### Adding a column named departure_date

In [65]:
powerbi_flights["departure_date"] = (
    powerbi_flights["departure_time"].dt.date
)

print("Departure date column created successfully.")
print("Date range:",powerbi_flights["departure_date"].min(),"to",powerbi_flights["departure_date"].max())

Departure date column created successfully.
Date range: 2026-04-17 to 2026-04-20


### Saving the dataset to be used in PowerBI in its respective .csv file

In [66]:
powerbi_flights.to_csv(
    "powerbi_flights.csv",
    index=False
)

print("Final Power BI dataset saved successfully.")
print("File: powerbi_flights.csv")
print("Records:", len(powerbi_flights))
print("Columns:", len(powerbi_flights.columns))

Final Power BI dataset saved successfully.
File: powerbi_flights.csv
Records: 1004
Columns: 9


In [67]:
data_dictionary = pd.DataFrame({
    "Column": powerbi_flights.columns,
    "Data Type": [str(powerbi_flights[col].dtype) for col in powerbi_flights.columns],
    "Description": [
        "Identifies a flight uniquely",
        "Airline name that is operating the flight",
        "Departure airport/city code",
        "Arrival airport/city code",
        "Flight departure timestamp",
        "Flight arrival timestamp",
        "Indicates whether the flight arrives on the next day(is an overnight flight)",
        "Calculated flight duration in minutes",
        "Date of flight departure"
    ]
})

print("Data dictionary created successfully.")
print(data_dictionary.to_string(index=False))

data_dictionary.to_csv(
    "data_dictionary.csv",
    index=False
)

print("Saved as: data_dictionary.csv")

Data dictionary created successfully.
          Column      Data Type                                                                  Description
       flight_id         string                                                 Identifies a flight uniquely
         airline         string                                    Airline name that is operating the flight
          source         string                                                  Departure airport/city code
     destination         string                                                    Arrival airport/city code
  departure_time datetime64[ns]                                                   Flight departure timestamp
    arrival_time datetime64[ns]                                                     Flight arrival timestamp
    is_overnight           bool Indicates whether the flight arrives on the next day(is an overnight flight)
duration_minutes        float64                                        Calculated flight d

### Whole pipeline summary- Data ingestion, cleaning, validation, transformation, and KPI generation

In [68]:
pipeline_summary = {
    "Source Flight Records": source_flight_count,
    "Flight Duplicates Removed": duplicate_count,
    "Clean Flight Records": len(clean_flights),
    "Rejected Flight Records": len(rejected_flights),
    "Average Flight Duration (minutes)": round(average_duration, 2),
    "Flight Anomaly Rate (%)": round(anomaly_rate, 2),
    "Total Bookings": total_bookings,
    "Booking Cancellation Rate (%)": round(cancellation_rate, 2),
    "Total Payment Revenue": round(total_payment_revenue, 2),
    "Average Payment Amount": round(average_payment_amount, 2),
    "Total Passengers": total_passengers
}

print("Pipeline completion summary")
logger.info("Pipeline completed successfully.")

for metric, value in pipeline_summary.items():
    print(f"{metric}: {value}")


Pipeline completion summary
Source Flight Records: 1020
Flight Duplicates Removed: 15
Clean Flight Records: 1004
Rejected Flight Records: 1
Average Flight Duration (minutes): 164.49
Flight Anomaly Rate (%): 0.1
Total Bookings: 924
Booking Cancellation Rate (%): 33.98
Total Payment Revenue: 6959344.65
Average Payment Amount: 8064.13
Total Passengers: 1000


## Cleaned Datase Ready for visualization - "powerbi_flights.csv"


##  Cleaned and validated flight records- "clean_flights.csv"


## Rejected/anomalous flight records with rejection reasons- "rejected_flights.csv"

## Overall business KPI summary- "kpi_summary.csv"



## Flight count by source-destination route- "route_traffic.csv"

## Flight count by airline "airline_distribution.csv"

## Column names, data types, and descriptions- "data_dictionary.csv"